# TCC - Analise Preditiva de Risco Academico Escolar
## CRISP-DM: 5. Evaluation

## O que esta fase avalia

A avaliacao verifica se a modelagem produziu sinais uteis para a escola:

- se a previsao de nota erra pouco;
- se o alerta de risco encontra alunos que realmente ficaram abaixo de 6.0;
- se o candidato escolhido supera baselines simples;
- se ha disciplinas, series ou faixas de nota com erro mais alto;
- se os resultados sao coerentes para apoiar professores, coordenacao e secretaria.

Nesta fase, a avaliacao deve ser lida como verificacao do desempenho no conjunto de teste final, depois que a escolha dos candidatos ja foi feita na validacao.

## Carregar bibliotecas e localizar o projeto

In [ ]:
from pathlib import Path
import re
import sys

import pandas as pd
from IPython.display import display

In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "school_predictor").exists() and (candidate / "artifacts").exists():
            return candidate.resolve()
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## Importar caminhos e execucao opcional

A avaliacao normalmente usa artefatos ja gerados pela fase de Modeling. Se eles nao existirem, e possivel rodar a pipeline localmente ativando uma flag.

In [ ]:
from school_predictor.pipeline.config import PipelinePaths
from school_predictor.pipeline.orchestration import resolve_mode_settings, run_real_pipeline

## Escolher modo avaliado

Os dois modos podem ser avaliados:

- `previsao_nota`: foco principal na nota prevista.
- `alerta_risco`: foco principal no alerta pedagogico.

Ambos geram metricas de regressao e classificacao, pois a pipeline calcula os dois sinais de forma conjunta.

O que muda entre os modos e principalmente o `min_history`, o volume do dataset e os artefatos salvos. A logica de avaliacao continua analisando nota prevista e risco previsto no teste final.

In [ ]:
MODE = "previsao_nota"  # alternativas: "previsao_nota" ou "alerta_risco"
MODE, MIN_HISTORY = resolve_mode_settings(MODE, min_history=None)
paths = PipelinePaths.from_project_root(PROJECT_ROOT, min_history=MIN_HISTORY, mode=MODE)

paths.output_dir

### O que este modo muda na avaliacao

A celula acima define qual conjunto de artefatos sera lido. Isso importa porque `previsao_nota` e `alerta_risco` podem ter datasets e candidatos vencedores diferentes, mesmo compartilhando a mesma logica geral de avaliacao.

## Opcional: gerar artefatos se ainda nao existirem

Por padrao, este notebook nao treina nem salva nada. Se os arquivos de avaliacao nao existirem localmente, altere `RUN_PIPELINE_IF_MISSING = True`.

In [ ]:
RUN_PIPELINE_IF_MISSING = False

required_artifacts = [
    paths.predictions_csv,
    paths.metrics_txt,
    paths.error_by_subject_csv,
    paths.error_by_series_csv,
    paths.error_by_band_csv,
]

missing = [path for path in required_artifacts if not path.exists()]
if missing and RUN_PIPELINE_IF_MISSING:
    run_real_pipeline(project_root=PROJECT_ROOT, mode=MODE, min_history=MIN_HISTORY)
elif missing:
    print("Artefatos ausentes. Rode a pipeline pela CLI ou ative RUN_PIPELINE_IF_MISSING localmente.")
    for path in missing:
        print(path)

## Carregar artefatos de avaliacao

A tabela de predicoes contem o teste final. As tabelas de erro agrupam os resultados por disciplina, serie e faixa de nota.

Em outras palavras: a validacao serviu para escolher os candidatos; o que este notebook inspeciona agora e o desempenho final no ano de teste.

### Exemplo fixo dos artefatos usados na avaliacao

A avaliação desta fase consulta principalmente estes artefatos gerados pela modelagem:

| modo | arquivo | papel_na_avaliacao |
| --- | --- | --- |
| previsao_nota | student_prediction_predictions.csv | comparar nota real, nota prevista e risco previsto no teste |
| previsao_nota | student_prediction_report.txt | mostrar ranking de candidatos e métricas finais |
| previsao_nota | error_analysis_by_subject.csv | localizar disciplinas com maior erro |
| previsao_nota | error_analysis_by_series.csv | comparar desempenho por série |
| previsao_nota | error_analysis_by_score_band.csv | comparar erro por faixa de nota |
| alerta_risco | student_prediction_predictions.csv | avaliar classificação de risco no conjunto de teste |

In [ ]:
predictions = pd.read_csv(paths.predictions_csv, encoding="utf-8", low_memory=False)
error_by_subject = pd.read_csv(paths.error_by_subject_csv, encoding="utf-8", low_memory=False)
error_by_series = pd.read_csv(paths.error_by_series_csv, encoding="utf-8", low_memory=False)
error_by_band = pd.read_csv(paths.error_by_band_csv, encoding="utf-8", low_memory=False)
report_text = paths.metrics_txt.read_text(encoding="utf-8")

pd.DataFrame([{
    "modo": MODE,
    "predicoes": len(predictions),
    "disciplinas_avaliadas": len(error_by_subject),
    "series_avaliadas": len(error_by_series),
    "faixas_avaliadas": len(error_by_band),
}])

### O que realmente entrou na avaliacao

Depois desta etapa, o notebook deixa de olhar para o dataset de treino e passa a olhar para os artefatos do teste final. A tabela de predições representa o que o modelo fez no ano mais recente; as tabelas agrupadas mostram onde ele foi melhor ou pior.

Essa diferenca e crucial: aqui ja nao estamos escolhendo candidato. Estamos auditando o comportamento do vencedor.

## Interpretacao das metricas

| Metrica | Onde aparece | Como interpretar |
|---|---|---|
| MAE | Regressao | erro medio absoluto da nota; quanto menor, melhor |
| RMSE | Regressao | penaliza erros grandes; quanto menor, melhor |
| Acerto <= 0.5 | Regressao | percentual de previsoes com erro de ate meio ponto |
| Precision | Classificacao | dos alertas emitidos, quantos eram risco real |
| Recall | Classificacao | dos riscos reais, quantos foram encontrados |
| F1 | Classificacao | equilibrio entre precision e recall |

No contexto pedagogico, recall baixo pode ser perigoso porque indica que alunos em risco deixaram de ser alertados. Precision muito baixo tambem e ruim porque gera muitos alertas falsos e pode cansar a equipe.

Por isso, este projeto nao usa apenas acuracia geral. Em bases escolares, um modelo pode acertar muitos casos triviais e ainda assim falhar justamente nos alunos que mais importam.

## Extrair resumo tecnico do relatorio

O relatorio tecnico salvo pela pipeline registra quem venceu na validacao e quais metricas esse vencedor obteve no teste final.

### Exemplo fixo do resumo tecnico

Abaixo está um resumo mais legível das métricas finais e da referência usada para comparação:

| modo | tarefa | metrica_principal | modelo_final | valor_modelo | baseline_referencia | valor_baseline |
| --- | --- | --- | --- | --- | --- | --- |
| previsao_nota | regressao | MAE | RandomForest | 0.7659 | ultima_nota | 0.9138 |
| previsao_nota | classificacao | F1 | Baseline media_duas_ultimas | 0.7350 | baseline final | 0.7252 |
| alerta_risco | regressao | MAE | HistGradientBoosting | 0.8140 | ultima_nota | 0.9405 |
| alerta_risco | classificacao | F1 | HistGradientBoosting | 0.7436 | baseline final | 0.7243 |

In [ ]:
def parse_metric_line(pattern: str, text: str) -> float | None:
    match = re.search(pattern, text)
    return float(match.group(1)) if match else None


selected_candidates = pd.DataFrame([
    {"tipo": "regressao", "candidato": re.search(r"candidato selecionado para regressao: ([^\n]+)", report_text).group(1)},
    {"tipo": "classificacao", "candidato": re.search(r"candidato selecionado para classificacao: ([^\n]+)", report_text).group(1)},
])

metric_summary = pd.DataFrame([
    {"grupo": "modelo_regressao", "metrica": "mae", "valor": parse_metric_line(r"modelo mae: ([0-9.]+)", report_text)},
    {"grupo": "modelo_regressao", "metrica": "rmse", "valor": parse_metric_line(r"modelo rmse: ([0-9.]+)", report_text)},
    {"grupo": "modelo_regressao", "metrica": "acerto_ate_0_5", "valor": parse_metric_line(r"modelo acerto <= 0.5: ([0-9.]+)", report_text)},
    {"grupo": "baseline_regressao", "metrica": "mae", "valor": parse_metric_line(r"baseline mae: ([0-9.]+)", report_text)},
    {"grupo": "baseline_regressao", "metrica": "rmse", "valor": parse_metric_line(r"baseline rmse: ([0-9.]+)", report_text)},
    {"grupo": "baseline_regressao", "metrica": "acerto_ate_0_5", "valor": parse_metric_line(r"baseline acerto <= 0.5: ([0-9.]+)", report_text)},
    {"grupo": "modelo_classificacao", "metrica": "precision", "valor": parse_metric_line(r"modelo precision: ([0-9.]+)", report_text)},
    {"grupo": "modelo_classificacao", "metrica": "recall", "valor": parse_metric_line(r"modelo recall: ([0-9.]+)", report_text)},
    {"grupo": "modelo_classificacao", "metrica": "f1", "valor": parse_metric_line(r"modelo f1: ([0-9.]+)", report_text)},
    {"grupo": "baseline_classificacao", "metrica": "precision", "valor": parse_metric_line(r"baseline precision: ([0-9.]+)", report_text)},
    {"grupo": "baseline_classificacao", "metrica": "recall", "valor": parse_metric_line(r"baseline recall: ([0-9.]+)", report_text)},
    {"grupo": "baseline_classificacao", "metrica": "f1", "valor": parse_metric_line(r"baseline f1: ([0-9.]+)", report_text)},
])

display(selected_candidates)
metric_summary

### Como ler o resumo tecnico extraido do relatorio

A saida acima concentra o que a pipeline registrou como desempenho final no teste. Ela junta, em formato compacto, o candidato escolhido e as metricas do modelo e do baseline.

Na pratica, esta e a visao que responde duas perguntas de defesa:
- quem venceu na validacao?
- no teste final, esse vencedor realmente entregou ganho util?

## Avaliacao direta na tabela de predicoes

Este bloco recalcula metricas simples a partir do arquivo de predicoes para conferir os resultados do relatorio.

A ideia aqui nao e substituir o relatorio tecnico, mas mostrar que os valores podem ser reproduzidos diretamente do conjunto de teste salvo em CSV.

In [ ]:
tp = ((predictions["target_risco_proxima"] == 1) & (predictions["predicted_risk_flag"] == 1)).sum()
fp = ((predictions["target_risco_proxima"] == 0) & (predictions["predicted_risk_flag"] == 1)).sum()
fn = ((predictions["target_risco_proxima"] == 1) & (predictions["predicted_risk_flag"] == 0)).sum()

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

direct_metrics = pd.DataFrame([{
    "mae_recalculado": predictions["absolute_error"].mean(),
    "rmse_recalculado": (predictions["signed_error"].pow(2).mean()) ** 0.5,
    "acerto_ate_0_5_recalculado": (predictions["absolute_error"] <= 0.5).mean(),
    "precision_risco_recalculada": precision,
    "recall_risco_recalculado": recall,
    "f1_risco_recalculado": f1,
    "acuracia_risco_recalculada": predictions["risk_hit"].mean(),
    "taxa_risco_real": predictions["target_risco_proxima"].mean(),
    "taxa_risco_previsto": predictions["predicted_risk_flag"].mean(),
}])

direct_metrics

### O que esta conferencia prova

A tabela recalculada mostra que as metricas podem ser reproduzidas diretamente do CSV de predições. Isso e importante porque demonstra rastreabilidade: o relatorio tecnico nao e uma caixa-preta separada dos dados.

Quando os numeros batem, a avaliacao ganha credibilidade metodologica.

## Amostra segura de predicoes

A amostra abaixo usa os nomes já pseudonimizados da base pública e mostra apenas os campos pedagógicos essenciais. Ainda assim, evite salvar outputs executados no GitHub quando não forem necessários.

### Exemplo fixo da tabela de predicoes

Esta amostra mostra como uma linha do teste é avaliada na prática, usando os nomes já pseudonimizados da base pública:

| NomeAluno | NomePeriodo | NomeSerie | NomeDisciplina | NomeEtapa | ValorMedia | target_nota_proxima | predicted_next_grade | absolute_error | target_risco_proxima | predicted_risk_flag | risk_hit |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Aluno 01790 | 2025 | 1ª Série | Arte | 2º BIMESTRE | 8.8 | 8.1 | 8.9894 | 0.8894 | 0 | 0 | 1 |
| Aluno 01790 | 2025 | 1ª Série | Arte | 3º BIMESTRE | 8.1 | 10.0 | 9.0035 | 0.9965 | 0 | 0 | 1 |
| Aluno 01790 | 2025 | 1ª Série | Biologia | 2º BIMESTRE | 7.0 | 7.4 | 7.7188 | 0.3188 | 0 | 0 | 1 |
| Aluno 01790 | 2025 | 1ª Série | Biologia | 3º BIMESTRE | 7.4 | 7.0 | 8.0697 | 1.0697 | 0 | 0 | 1 |
| Aluno 01790 | 2025 | 1ª Série | Biologia 1 | 2º BIMESTRE | 7.0 | 5.6 | 7.4285 | 1.8285 | 1 | 0 | 0 |

In [ ]:
public_prediction_columns = [
    "NomePeriodo", "NomeSerie", "NomeDisciplina", "NomeEtapa",
    "ValorMedia", "target_nota_proxima", "predicted_next_grade",
    "absolute_error", "target_risco_proxima", "predicted_risk_flag", "risk_hit",
]

predictions[[column for column in public_prediction_columns if column in predictions.columns]].head(10)

### O que esta amostra de predicoes revela

Agora voce enxerga uma linha real de avaliacao: nota atual, alvo da proxima nota, nota prevista, risco real, risco previsto e erro. Isso ajuda a banca a sair do nivel abstrato das metricas e ver como o modelo se comporta caso a caso.

## Analise de erro por disciplina

Esta tabela mostra onde o erro medio de nota foi maior. Ela ajuda a identificar disciplinas que podem precisar de tratamento especifico, mais dados ou avaliacao pedagogica mais cuidadosa.

### Exemplo fixo do erro por disciplina

As disciplinas abaixo são as que concentraram maior erro médio absoluto no modo `previsao_nota`:

| NomeDisciplina | samples | mae | risk_accuracy |
| --- | --- | --- | --- |
| Matemática 3 | 163 | 1.5887 | 0.6871 |
| Matemática 4 e 5 | 163 | 1.4575 | 0.8589 |
| Biologia 2 | 521 | 1.3515 | 0.7006 |
| Biologia 1 | 521 | 1.1428 | 0.7831 |
| Matemática 1 e 2 | 163 | 1.1357 | 0.8957 |

In [ ]:
error_by_subject.head(15)

### O que o erro por disciplina mostra

Quando esta tabela aparece, a avaliacao deixa de ser global. Ela passa a apontar onde o modelo sofre mais. Isso e essencial em contexto escolar, porque um MAE medio bom pode esconder disciplinas especificas com comportamento bem pior.

## Analise de erro por serie

Avaliacao por serie ajuda a saber se o modelo funciona melhor em certos anos escolares e pior em outros.

### Exemplo fixo do erro por serie

Aqui a avaliação mostra quais séries ficaram mais difíceis de prever:

| NomeSerie | samples | mae | risk_accuracy |
| --- | --- | --- | --- |
| 2ª Série | 4756 | 0.8739 | 0.8522 |
| 9º Ano | 3698 | 0.8701 | 0.8437 |
| 3ª Série | 5053 | 0.8398 | 0.8698 |
| 1ª Série | 5432 | 0.8294 | 0.8575 |
| 6º Ano | 2610 | 0.5864 | 0.9479 |

In [ ]:
error_by_series.head(15)

### O que o erro por serie mostra

Aqui a pergunta muda de disciplina para etapa escolar. A saida ajuda a enxergar se o sistema funciona de modo desigual entre anos escolares, o que pode indicar necessidade de novos atributos ou leitura pedagogica mais cuidadosa em certas series.

## Analise de erro por faixa de nota

A avaliacao por faixa mostra se o modelo erra mais em alunos de nota baixa, media ou alta. Isso e importante porque erros em notas proximas de 6.0 podem mudar a decisao de alerta.

### Exemplo fixo do erro por faixa de nota

Esta tabela ajuda a ver se o erro muda quando a nota real está em faixas diferentes:

| score_band | samples | mae | risk_accuracy |
| --- | --- | --- | --- |
| abaixo_6 | 5756 | 1.031 | 0.751 |
| 6_a_6_99 | 3063 | 0.877 | 0.6941 |
| 7_a_7_99 | 3476 | 0.766 | 0.8708 |
| 8_a_8_99 | 4220 | 0.6858 | 0.9491 |
| 9_ou_mais | 10012 | 0.6134 | 0.9916 |

In [ ]:
error_by_band

### O que a faixa de nota revela

Esta analise mostra se o modelo erra mais justamente nos alunos mais sensiveis. Em projetos educacionais, isso importa muito, porque pequenas diferencas perto da linha de 6.0 podem mudar completamente a interpretacao do risco.

## Matriz simples de risco

A matriz abaixo compara risco real e risco previsto. Ela ajuda a visualizar falsos positivos e falsos negativos.

Leitura rapida:
- falso negativo e aluno em risco real que o sistema nao alertou;
- falso positivo e aluno alertado que nao entrou em risco na nota seguinte.

No contexto escolar deste projeto, falsos negativos costumam ser mais preocupantes do que falsos positivos moderados.

### Exemplo fixo da matriz de risco

A matriz abaixo resume acertos e erros da classificação de risco no teste:

| risco_real | previsto_sem_risco | previsto_com_risco |
| --- | --- | --- |
| real_sem_risco | 19086 | 1685 |
| real_com_risco | 1433 | 4323 |

In [ ]:
risk_matrix = pd.crosstab(
    predictions["target_risco_proxima"],
    predictions["predicted_risk_flag"],
    rownames=["risco_real"],
    colnames=["risco_previsto"],
    margins=True,
)

risk_matrix

### O que a matriz de risco deixa visivel

A matriz transforma a classificacao em contagens concretas. Depois dela, fica facil explicar quantos casos foram:
- acertos de risco;
- acertos de nao risco;
- falsos positivos;
- falsos negativos.

Essa leitura ajuda a traduzir precision, recall e F1 em impacto operacional para a escola.

## Criterios para dizer se esta bom

Nao existe um unico numero universal. Para este projeto, a avaliacao deve considerar:

- MAE menor que o baseline indica ganho real sobre regra simples.
- RMSE muito maior que MAE indica alguns erros grandes que precisam ser investigados.
- Acerto ate 0.5 ponto ajuda a saber se a previsao e pedagogicamente proxima.
- F1 maior que o baseline indica que a classificacao aprendeu algo alem de regras simples.
- Recall baixo exige cautela, pois significa deixar alunos em risco sem alerta.
- Precision baixo exige cautela, pois gera muitos alertas falsos para professores e coordenacao.

Tambem e importante observar qual baseline foi usado como referencia no modo atual. Em alguns cenarios, o baseline vencedor na validacao nao e necessariamente `ultima_nota`, mas outra regra simples como `media_duas_ultimas`.

## Resultado da fase

Ao final da avaliacao, o projeto deve conseguir explicar:

- qual candidato foi escolhido na validacao;
- se ele superou os baselines no teste final;
- quais foram os erros medios de nota;
- como o alerta de risco se comportou;
- onde o modelo erra mais;
- quais cuidados pedagogicos devem acompanhar o uso dos relatorios.

Essas conclusoes seguem para a fase de Deployment, onde os resultados sao transformados em relatorios e dashboard para os perfis da escola.

A avaliacao, portanto, nao responde apenas se o modelo acertou ou errou. Ela responde se o sinal produzido e suficientemente confiavel para entrar em um fluxo escolar real sem criar falsa seguranca nem ruído excessivo.

## Leitura didatica da avaliacao

O resultado final desta fase nao e apenas dizer que o modelo ficou "bom" ou "ruim". O que sai daqui e uma interpretacao defensavel sobre tres dimensoes ao mesmo tempo:
- ganho sobre baseline;
- comportamento por grupos criticos;
- utilidade institucional do sinal gerado.